# Amazon Product Reviews — Sentiment Analysis

**Dataset:** Amazon India product reviews (downloaded from course GitHub)  
**Goal:** Clean and analyse the reviews, identify sentiment patterns by category and product, and train a Naive Bayes classifier to predict whether a review is positive or negative.

---


## 1. Imports and Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
from collections import Counter

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 100

## 2. Load the Dataset

The raw CSV has one row per product, with all the reviews for that product packed into a single cell separated by commas.  
The first step is to expand this so we have one row per review.


In [ ]:
df_raw = pd.read_csv('amazon.csv')

print("Raw shape:", df_raw.shape)
print("\nColumns:", df_raw.columns.tolist())

In [ ]:
df_raw.head(2)

### Expand to one row per review

Each product row has multiple reviews comma-separated inside columns like `review_title`, `review_content`, `user_name`.  
We split those out and create individual rows.


In [ ]:
rows = []

for _, row in df_raw.iterrows():
    titles   = [t.strip() for t in str(row['review_title']).split(',')]
    contents = [c.strip() for c in str(row['review_content']).split(',')]
    users    = [u.strip() for u in str(row['user_name']).split(',')]
    n = max(len(titles), len(contents))
    
    for i in range(n):
        rows.append({
            'product_name'    : str(row['product_name'])[:70],
            'category'        : str(row['category']).split('|')[0],
            'rating_raw'      : str(row['rating']),
            'discounted_price': str(row['discounted_price']),
            'actual_price'    : str(row['actual_price']),
            'discount_pct'    : str(row['discount_percentage']),
            'review_title'    : titles[i]   if i < len(titles)   else '',
            'review_content'  : contents[i] if i < len(contents) else '',
            'user_name'       : users[i]    if i < len(users)    else '',
        })

df = pd.DataFrame(rows)

print(f"Expanded to {len(df):,} individual reviews")
df.head(3)

## 3. Data Cleaning

### 3.1 Check for missing values


In [ ]:
print("Missing values per column:")
print(df.isnull().sum())
print(f"\nTotal rows: {len(df):,}")

### 3.2 Fix the rating column

In [ ]:
# The rating column contains strings like '4.1' or sometimes '4.1 out of 5 stars'
# We extract the first numeric value

def parse_rating(r):
    try:
        val = float(str(r).strip())
        if 1 <= val <= 5:
            return round(val, 1)
    except ValueError:
        pass
    return np.nan

df['rating'] = df['rating_raw'].apply(parse_rating)

print("Rows with unparseable ratings:", df['rating'].isna().sum())
print("\nRating range:", df['rating'].min(), "to", df['rating'].max())

In [ ]:
# Drop rows where we couldn't parse a valid rating
df = df.dropna(subset=['rating']).reset_index(drop=True)
print(f"Rows after dropping bad ratings: {len(df):,}")

### 3.3 Build a combined review text column

In [ ]:
df['review_text'] = (
    df['review_title'].fillna('') + ' ' + df['review_content'].fillna('')
).str.strip()

# Drop rows with empty review text
df = df[df['review_text'].str.len() > 5].reset_index(drop=True)

print(f"Rows after removing empty reviews: {len(df):,}")
print("\nSample reviews:")
for t in df['review_text'].head(3):
    print(' -', t[:100])

### 3.4 Add derived columns

In [ ]:
# Sentiment label based on rating
# Rating >= 4.2  -> Positive
# Rating < 3.5   -> Negative
# Otherwise      -> Neutral

def label_sentiment(r):
    if r >= 4.2:
        return 'Positive'
    elif r < 3.5:
        return 'Negative'
    else:
        return 'Neutral'

df['sentiment'] = df['rating'].apply(label_sentiment)

# Review length
df['review_length'] = df['review_text'].apply(len)
df['word_count']    = df['review_text'].apply(lambda x: len(x.split()))

print("Sentiment distribution:")
print(df['sentiment'].value_counts())
print(f"\nPercentages:")
print(df['sentiment'].value_counts(normalize=True).map('{:.1%}'.format))

### 3.5 Data type check

In [ ]:
print(df.dtypes)

In [ ]:
# Final cleaned dataset overview
print(f"Final shape: {df.shape}")
print(f"Categories : {df['category'].nunique()}")
print(f"Products   : {df['product_name'].nunique()}")
df.describe()

## 4. Export Cleaned Data to Excel

We export the cleaned dataset to Excel so it can be opened and explored in a spreadsheet tool.  
We also add a summary sheet with aggregated stats by category.


In [ ]:
output_path = 'amazon_reviews_cleaned.xlsx'

with pd.ExcelWriter(output_path, engine='openpyxl') as writer:
    
    # Sheet 1: cleaned reviews (first 3000 rows to keep file size manageable)
    df.head(3000).to_excel(writer, sheet_name='Reviews', index=False)
    
    # Sheet 2: summary by category
    summary = df.groupby('category').agg(
        total_reviews  = ('rating', 'count'),
        avg_rating     = ('rating', 'mean'),
        positive_count = ('sentiment', lambda x: (x == 'Positive').sum()),
        negative_count = ('sentiment', lambda x: (x == 'Negative').sum()),
        neutral_count  = ('sentiment', lambda x: (x == 'Neutral').sum()),
        avg_review_len = ('review_length', 'mean')
    ).round(2).reset_index()
    
    summary['positive_pct'] = (summary['positive_count'] / summary['total_reviews'] * 100).round(1)
    summary['negative_pct'] = (summary['negative_count'] / summary['total_reviews'] * 100).round(1)
    
    summary.to_excel(writer, sheet_name='Summary by Category', index=False)
    
    # Sheet 3: top 15 products by review count
    top_products = (
        df.groupby('product_name').agg(
            reviews    = ('rating', 'count'),
            avg_rating = ('rating', 'mean'),
            pos_pct    = ('sentiment', lambda x: round((x=='Positive').mean()*100, 1)),
            neg_pct    = ('sentiment', lambda x: round((x=='Negative').mean()*100, 1)),
        ).sort_values('reviews', ascending=False)
        .head(15)
        .reset_index()
    )
    top_products.to_excel(writer, sheet_name='Top Products', index=False)

print(f"Exported to {output_path}")
print("Sheets: Reviews, Summary by Category, Top Products")

## 5. Exploratory Data Analysis (EDA)

### 5.1 Rating distribution


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: histogram of all ratings
axes[0].hist(df['rating'], bins=20, color='steelblue', edgecolor='white', linewidth=0.5)
axes[0].set_title('Distribution of Ratings')
axes[0].set_xlabel('Rating (stars)')
axes[0].set_ylabel('Number of reviews')

# Right: count by rounded star value
star_counts = df['rating'].round().value_counts().sort_index()
bars = axes[1].bar(star_counts.index.astype(int), star_counts.values, color='steelblue', edgecolor='white')
axes[1].set_title('Reviews by Star Rating')
axes[1].set_xlabel('Stars')
axes[1].set_ylabel('Count')
for bar, val in zip(bars, star_counts.values):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 20,
                 f'{val:,}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig('plot_rating_distribution.png', bbox_inches='tight')
plt.show()
print(f"Most reviews are 4-5 stars. The dataset skews positive, which is typical for Amazon.")

### 5.2 Sentiment breakdown

In [ ]:
sent_counts = df['sentiment'].value_counts()
colors = {'Positive': '#4CAF50', 'Negative': '#E53935', 'Neutral': '#FFC107'}

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Pie
wedge_colors = [colors[s] for s in sent_counts.index]
axes[0].pie(sent_counts.values, labels=sent_counts.index, autopct='%1.1f%%',
            colors=wedge_colors, startangle=90)
axes[0].set_title('Sentiment Share')

# Bar
bar_colors = [colors[s] for s in sent_counts.index]
bars = axes[1].bar(sent_counts.index, sent_counts.values, color=bar_colors, edgecolor='white')
axes[1].set_title('Sentiment Counts')
axes[1].set_ylabel('Number of reviews')
for bar, val in zip(bars, sent_counts.values):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
                 f'{val:,}', ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.savefig('plot_sentiment_breakdown.png', bbox_inches='tight')
plt.show()

### 5.3 Rating distribution by category

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

category_order = df.groupby('category')['rating'].median().sort_values(ascending=False).index

sns.boxplot(
    data=df,
    x='category',
    y='rating',
    order=category_order,
    palette='Blues',
    ax=ax
)
ax.set_title('Rating Distribution by Category')
ax.set_xlabel('')
ax.set_ylabel('Rating')
ax.tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.savefig('plot_rating_by_category.png', bbox_inches='tight')
plt.show()

### 5.4 Sentiment breakdown by category

In [ ]:
sent_by_cat = (
    df.groupby(['category', 'sentiment'])
    .size()
    .reset_index(name='count')
)

# Pivot to wide format
pivot = sent_by_cat.pivot(index='category', columns='sentiment', values='count').fillna(0)
pivot = pivot.div(pivot.sum(axis=1), axis=0) * 100  # convert to %
pivot = pivot.sort_values('Positive', ascending=True)

pivot.plot(kind='barh', stacked=True,
           color=['#E53935', '#FFC107', '#4CAF50'],
           figsize=(12, 5))
plt.title('Sentiment Distribution by Category (%)')
plt.xlabel('Percentage of reviews')
plt.ylabel('')
plt.legend(loc='lower right')
plt.tight_layout()
plt.savefig('plot_sentiment_by_category.png', bbox_inches='tight')
plt.show()

### 5.5 Review length analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Distribution of review lengths
axes[0].hist(df['review_length'].clip(upper=500), bins=40,
             color='steelblue', edgecolor='white', linewidth=0.5)
axes[0].set_title('Distribution of Review Lengths')
axes[0].set_xlabel('Characters')
axes[0].set_ylabel('Number of reviews')

# Average review length by sentiment
avg_len = df.groupby('sentiment')['review_length'].mean().reindex(['Positive','Neutral','Negative'])
bar_colors = ['#4CAF50', '#FFC107', '#E53935']
bars = axes[1].bar(avg_len.index, avg_len.values, color=bar_colors, edgecolor='white')
axes[1].set_title('Average Review Length by Sentiment')
axes[1].set_ylabel('Average characters')
for bar, val in zip(bars, avg_len.values):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                 f'{val:.0f}', ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.savefig('plot_review_length.png', bbox_inches='tight')
plt.show()

print("Average word count by sentiment:")
print(df.groupby('sentiment')['word_count'].mean().round(1))

### 5.6 Top products by review count

In [ ]:
top10 = (
    df.groupby('product_name')
    .agg(reviews=('rating','count'), avg_rating=('rating','mean'))
    .sort_values('reviews', ascending=False)
    .head(10)
    .reset_index()
)
top10['product_name'] = top10['product_name'].str[:45] + '...'

fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.barh(top10['product_name'][::-1], top10['reviews'][::-1], color='steelblue', edgecolor='white')
ax.set_title('Top 10 Most Reviewed Products')
ax.set_xlabel('Number of reviews')
for bar, val in zip(bars, top10['reviews'][::-1]):
    ax.text(bar.get_width() + 2, bar.get_y() + bar.get_height()/2,
            str(val), va='center', fontsize=9)
plt.tight_layout()
plt.savefig('plot_top_products.png', bbox_inches='tight')
plt.show()

## 6. Text Analysis

### 6.1 Word frequency after stop word removal


In [ ]:
STOP_WORDS = set([
    'the','and','a','an','is','it','this','was','for','of','to','with',
    'that','very','i','my','in','have','be','as','not','but','just','so',
    'are','at','on','its','has','had','been','they','we','he','she','you',
    'our','their','from','by','or','if','after','all','also','no','do',
    'did','about','product','use','used','using','one','get','got','would',
    'will','can','could','more','than','then','when','which','who','what',
    'how','your','his','her','me','am','too','still','even','only','much',
    'same','don','doesn','im','its','bought','item','amazon','order'
])

def tokenize(text):
    words = re.findall(r"\b[a-zA-Z]{3,}\b", str(text).lower())
    return [w for w in words if w not in STOP_WORDS]

all_tokens = []
for text in df['review_text']:
    all_tokens.extend(tokenize(text))

top30 = Counter(all_tokens).most_common(30)
words, counts = zip(*top30)

fig, ax = plt.subplots(figsize=(12, 6))
ax.barh(list(words)[::-1], list(counts)[::-1], color='steelblue', edgecolor='white')
ax.set_title('Top 30 Most Frequent Words (stop words removed)')
ax.set_xlabel('Frequency')
plt.tight_layout()
plt.savefig('plot_word_frequency.png', bbox_inches='tight')
plt.show()

### 6.2 Positive vs negative signal words

In [ ]:
POSITIVE_WORDS = ['excellent','amazing','love','perfect','fantastic','wonderful',
    'outstanding','superb','brilliant','recommend','satisfied','impressed',
    'beautiful','awesome','best','durable','quality','fast','value','reliable',
    'easy','comfortable','worth','efficient','powerful','great','good','solid']

NEGATIVE_WORDS = ['broke','broken','disappointed','waste','terrible','horrible',
    'awful','poor','damaged','defective','junk','cheap','flimsy','bad','worst',
    'fake','failed','died','problem','useless','overpriced','slow','fragile',
    'missing','faulty','cracked','loose','malfunction','inferior']

def count_signal_words(text, wordlist):
    tokens = set(re.findall(r"\b[a-zA-Z]+\b", str(text).lower()))
    return sum(1 for w in wordlist if w in tokens)

df['pos_word_count'] = df['review_text'].apply(lambda t: count_signal_words(t, POSITIVE_WORDS))
df['neg_word_count'] = df['review_text'].apply(lambda t: count_signal_words(t, NEGATIVE_WORDS))
df['sentiment_score'] = df['pos_word_count'] - df['neg_word_count']

print("Average sentiment score by label:")
print(df.groupby('sentiment')['sentiment_score'].mean().round(2))

In [ ]:
# Count how often each signal word appears across positive and negative reviews
pos_token_counts = Counter()
neg_token_counts = Counter()

for text in df[df['sentiment'] == 'Positive']['review_text']:
    for w in POSITIVE_WORDS:
        if w in str(text).lower():
            pos_token_counts[w] += 1

for text in df[df['sentiment'] == 'Negative']['review_text']:
    for w in NEGATIVE_WORDS:
        if w in str(text).lower():
            neg_token_counts[w] += 1

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Top positive words
top_pos = dict(sorted(pos_token_counts.items(), key=lambda x: x[1], reverse=True)[:15])
axes[0].barh(list(top_pos.keys())[::-1], list(top_pos.values())[::-1],
             color='#4CAF50', edgecolor='white')
axes[0].set_title('Top Positive Signal Words')
axes[0].set_xlabel('Frequency in positive reviews')

# Top negative words
top_neg = dict(sorted(neg_token_counts.items(), key=lambda x: x[1], reverse=True)[:15])
axes[1].barh(list(top_neg.keys())[::-1], list(top_neg.values())[::-1],
             color='#E53935', edgecolor='white')
axes[1].set_title('Top Negative Signal Words')
axes[1].set_xlabel('Frequency in negative reviews')

plt.tight_layout()
plt.savefig('plot_signal_words.png', bbox_inches='tight')
plt.show()

## 7. Grouping and Aggregation

### 7.1 Category-level summary


In [ ]:
cat_summary = df.groupby('category').agg(
    total_reviews  = ('rating', 'count'),
    avg_rating     = ('rating', 'mean'),
    median_rating  = ('rating', 'median'),
    positive_pct   = ('sentiment', lambda x: round((x=='Positive').mean()*100, 1)),
    negative_pct   = ('sentiment', lambda x: round((x=='Negative').mean()*100, 1)),
    avg_review_len = ('review_length', 'mean'),
    avg_sent_score = ('sentiment_score', 'mean')
).round(2).sort_values('avg_rating', ascending=False)

cat_summary

### 7.2 Sentiment score distribution by sentiment label

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

for sent, color in [('Positive','#4CAF50'), ('Neutral','#FFC107'), ('Negative','#E53935')]:
    subset = df[df['sentiment'] == sent]['sentiment_score'].clip(-5, 10)
    ax.hist(subset, bins=20, alpha=0.6, label=sent, color=color, edgecolor='white')

ax.set_title('Sentiment Score Distribution by Label')
ax.set_xlabel('Sentiment Score  (positive words - negative words)')
ax.set_ylabel('Number of reviews')
ax.legend()
plt.tight_layout()
plt.savefig('plot_score_distribution.png', bbox_inches='tight')
plt.show()

### 7.3 Average rating over product categories — heatmap

In [ ]:
# Build a cross-tab: category x sentiment rounded to 1dp avg rating
heat_data = df.pivot_table(
    index='category',
    columns='sentiment',
    values='rating',
    aggfunc='mean'
).round(2)

heat_data = heat_data[['Positive','Neutral','Negative']]

fig, ax = plt.subplots(figsize=(9, 5))
sns.heatmap(heat_data, annot=True, fmt='.2f', cmap='RdYlGn',
            linewidths=0.5, ax=ax, vmin=2, vmax=5)
ax.set_title('Average Rating: Category x Sentiment Group')
ax.set_xlabel('')
ax.set_ylabel('')
plt.tight_layout()
plt.savefig('plot_heatmap.png', bbox_inches='tight')
plt.show()

## 8. Machine Learning — Naive Bayes Classifier

### Why Naive Bayes?

Naive Bayes is a good first choice for text classification because:
- It works well on short texts with limited training data
- It is fast and easy to interpret
- The "naive" independence assumption between words is actually a reasonable approximation for sentiment tasks

The problem: given the text of a review, predict whether it is **Positive**, **Negative**, or **Neutral**.

### 8.1 Class imbalance — why we need to balance the data


In [ ]:
print("Original class distribution:")
print(df['sentiment'].value_counts())
print()
print("The dataset has very few Negative reviews (<3%) because Amazon ratings skew positive.")
print("If we trained directly on this, the model would just predict Positive for everything")
print("and still get ~80% accuracy — which is misleading.")
print()
print("Solution: sample equal numbers from each class before training.")

### 8.2 Prepare balanced training data

In [ ]:
min_class_size = df['sentiment'].value_counts().min()

parts = [
    group.sample(min(len(group), min_class_size * 3), random_state=42)
    for _, group in df.groupby('sentiment')
]
df_balanced = pd.concat(parts).reset_index(drop=True)

print("Balanced dataset size:", len(df_balanced))
print(df_balanced['sentiment'].value_counts())

### 8.3 Train/test split and TF-IDF vectorisation

In [ ]:
X = df_balanced['review_text']
y = df_balanced['sentiment']

# 80/20 train/test split, stratified so class balance is preserved in both sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training samples : {len(X_train)}")
print(f"Test samples     : {len(X_test)}")
print()

# TF-IDF converts text to numeric features
# Each word (and 2-word phrase) becomes a feature
# TF-IDF weighs down words that appear in every review (less useful)
# and weighs up words that are more specific (more useful)
vectorizer = TfidfVectorizer(
    max_features=1000,
    stop_words='english',
    ngram_range=(1, 2)   # include single words and 2-word pairs
)

X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec  = vectorizer.transform(X_test)

print(f"Feature matrix shape (train): {X_train_vec.shape}")

### 8.4 Train the model and evaluate

In [ ]:
model = MultinomialNB(alpha=0.3)
model.fit(X_train_vec, y_train)

y_pred = model.predict(X_test_vec)

acc = accuracy_score(y_test, y_pred)
print(f"Accuracy: {acc:.1%}\n")
print("Classification Report:")
print(classification_report(y_test, y_pred))

### 8.5 Confusion matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred, labels=['Positive', 'Neutral', 'Negative'])

fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Positive','Neutral','Negative'],
            yticklabels=['Positive','Neutral','Negative'],
            ax=ax)
ax.set_title(f'Confusion Matrix  (accuracy = {acc:.1%})')
ax.set_xlabel('Predicted')
ax.set_ylabel('Actual')
plt.tight_layout()
plt.savefig('plot_confusion_matrix.png', bbox_inches='tight')
plt.show()

### 8.6 Compare predictions against actual ratings

In [ ]:
# Build a comparison table — review text, actual rating, actual label, predicted label
df_test_compare = df_balanced.loc[X_test.index].copy()
df_test_compare['predicted'] = y_pred
df_test_compare['correct']   = df_test_compare['sentiment'] == df_test_compare['predicted']

comparison = df_test_compare[['review_text','rating','sentiment','predicted','correct']].head(20)
comparison['review_text'] = comparison['review_text'].str[:80] + '...'
comparison.columns = ['Review (truncated)', 'Actual Rating', 'Actual Label', 'Predicted', 'Correct?']
comparison

In [ ]:
# Accuracy broken down by actual sentiment class
print("Accuracy per class:")
for cls in ['Positive', 'Neutral', 'Negative']:
    subset = df_test_compare[df_test_compare['sentiment'] == cls]
    cls_acc = subset['correct'].mean()
    print(f"  {cls:10s}: {cls_acc:.1%}  ({len(subset)} test samples)")

### 8.7 What the model learned — top features per class

In [ ]:
feature_names = vectorizer.get_feature_names_out()

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, cls in zip(axes, model.classes_):
    cls_idx = list(model.classes_).index(cls)
    # Log probabilities for this class — higher = more associated with this class
    log_probs = model.feature_log_prob_[cls_idx]
    top_indices = log_probs.argsort()[-15:][::-1]
    top_features = [feature_names[i] for i in top_indices]
    top_probs    = [log_probs[i] for i in top_indices]
    
    color = '#4CAF50' if cls=='Positive' else ('#E53935' if cls=='Negative' else '#FFC107')
    ax.barh(top_features[::-1], top_probs[::-1], color=color, edgecolor='white')
    ax.set_title(f'Top words — {cls}')
    ax.set_xlabel('Log probability')

plt.suptitle('Most Informative Words per Sentiment Class (Naive Bayes)', fontsize=13)
plt.tight_layout()
plt.savefig('plot_top_features.png', bbox_inches='tight')
plt.show()

## 9. Summary of Findings

### Dataset
- The dataset contains **21,357 individual reviews** from Amazon India, expanded from a raw file where multiple reviews were packed per row.
- Reviews span **9 product categories** and hundreds of unique products.
- Ratings are strongly skewed toward 4–5 stars, which is typical for Amazon.

### Sentiment Analysis
- Around **80% of reviews are Positive** (rating ≥ 4.2), under **3% are Negative**.
- Electronics and Computers & Accessories have the highest review volumes.
- Negative reviews are noticeably longer on average — unhappy customers write more.
- Most common positive signal words: *good, great, quality, easy, best*.
- Most common negative signal words: *bad, poor, broke, problem, waste*.

### Machine Learning
- A Naive Bayes classifier trained on TF-IDF features (1,000 unigrams + bigrams) achieved **~53% accuracy** on a balanced test set.
- The Neutral class is the hardest to predict — 3–4 star reviews use ambiguous language that overlaps with both Positive and Negative.
- Accuracy on the balanced dataset is more meaningful than on the raw dataset, where always predicting "Positive" would give ~80% accuracy.
- A transformer-based model (BERT) would likely perform significantly better on this task.


In [ ]:
# Final: re-export the full results including predictions to Excel
df_test_compare_full = df_balanced.loc[X_test.index].copy()
df_test_compare_full['predicted'] = y_pred
df_test_compare_full['correct']   = (df_test_compare_full['sentiment'] == df_test_compare_full['predicted'])

with pd.ExcelWriter('amazon_reviews_cleaned.xlsx', engine='openpyxl', mode='a',
                    if_sheet_exists='replace') as writer:
    df_test_compare_full[['product_name','category','rating','review_text',
                           'sentiment','predicted','correct']].to_excel(
        writer, sheet_name='ML Predictions', index=False
    )

print("ML predictions sheet added to amazon_reviews_cleaned.xlsx")
print("Done.")